# Monitor Xiaohongshu (RedNote) keywords and brand accounts

Take a snapshot of the notes Xiaohongshu (小红书, RedNote) returns for a keyword, plus the newest notes of a few brand accounts: likes, saves, comments and shares go into a CSV. Run it again later and it lists the new notes and the ones whose interactions grew fastest. No Xiaohongshu account or Chinese phone number needed.

The script is [`monitor.py`](https://github.com/okdataapi/xiaohongshu-keyword-monitor) (standard library only). It calls the [okdata](https://okdata.online/?src=colab-xiaohongshu-keyword-monitor) API: one page of about 20 notes is one call at $0.02. A run with one keyword and one account is 2 calls.

中文：输入小红书关键词和品牌号，记录笔记的赞、藏、评、转；再跑一次就能看到新笔记和互动涨得最快的笔记。把下面的 `LANG` 改成 `"zh"` 报告就是中文。

## 1. Your API key

Get a key at [okdata.online](https://okdata.online/signup/?src=colab-xiaohongshu-keyword-monitor) (new keys come with trial credit). In Colab, the easiest way is the key icon in the left sidebar (Secrets): add `OKDATA_API_KEY` and allow this notebook to read it. Otherwise the cell below asks for it. The key is kept in this session only.

In [ ]:
import os, getpass

def find_key():
    try:  # Colab secrets
        from google.colab import userdata
        return userdata.get("OKDATA_API_KEY")
    except Exception:
        pass
    try:  # Kaggle secrets
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("OKDATA_API_KEY")
    except Exception:
        pass
    return None

if not os.environ.get("OKDATA_API_KEY"):
    os.environ["OKDATA_API_KEY"] = find_key() or getpass.getpass("okdata API key (okd_...): ")
print("key loaded" if os.environ["OKDATA_API_KEY"].startswith("okd_") else "that does not look like an okdata key")

## 2. Get the script

In [ ]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/okdataapi/xiaohongshu-keyword-monitor/main/monitor.py", "monitor.py")
print("monitor.py downloaded")

## 3. Choose what to monitor

Keywords as you would type them into the Xiaohongshu search box (Chinese works best). Accounts by the 24-character id from `xiaohongshu.com/user/profile/<id>`, with `=NAME` to label them. `--dry-run` shows the plan and the most it can cost before anything is spent.

In [ ]:
import subprocess, sys

KEYWORDS = ["露营装备"]                                   # camping gear
ACCOUNTS = ["562f892fe00dd8540fd94691=迪卡侬DECATHLON"]   # id=label
SORT = "latest"   # latest, popular or general
PAGES = 1         # pages per keyword or account, about 20 notes each
LANG = "en"       # report language: en or zh

args = [a for k in KEYWORDS for a in ("--keyword", k)] + [a for c in ACCOUNTS for a in ("--account", c)]
args += ["--sort", SORT, "--pages", str(PAGES), "--lang", LANG]

def run(*extra):
    r = subprocess.run([sys.executable, "monitor.py", *args, *extra], capture_output=True, text=True)
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)

run("--dry-run")

## 4. Take a snapshot

In [ ]:
run()

## 5. Look at the data

In [ ]:
import pandas as pd

df = pd.read_csv("data/snapshots.csv")
latest = df[df.run_at == df.run_at.max()].copy()
latest["interactions"] = latest[["likes", "saves", "comments"]].fillna(0).sum(axis=1)
latest = latest.sort_values("interactions", ascending=False).reset_index(drop=True)
latest.index += 1
latest[["source", "author_name", "title", "likes", "saves", "comments", "shares", "published_at", "url"]].head(10)

In [ ]:
import matplotlib.pyplot as plt

# Numbered like the table above, so the chart needs no Chinese font.
top = latest.head(10).iloc[::-1]
labels = [f"#{i} {t} · {str(p)[:10]}" for i, t, p in zip(top.index, top.type, top.published_at)]
plt.figure(figsize=(9, 5))
plt.barh(labels, top.likes, label="likes")
plt.barh(labels, top.saves, left=top.likes, label="saves")
plt.barh(labels, top.comments, left=top.likes + top.saves, label="comments")
plt.xlabel("interactions (likes + saves + comments)")
plt.title("Notes with the most interactions in the latest snapshot")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Run it again later

Come back at least an hour later (or tomorrow) and run the snapshot cell again: the report then lists new notes and the biggest movers since the last run. A notebook session forgets its files when it ends, so for a lasting daily history fork the [repository](https://github.com/okdataapi/xiaohongshu-keyword-monitor) and let its GitHub Action take a snapshot every day.

The same Xiaohongshu data (note details, comments, accounts, topics) is also available to AI assistants such as Claude, Cursor and Dify through okdata's MCP server: see [okdata.online/mcp](https://okdata.online/mcp/?src=colab-xiaohongshu-keyword-monitor).